In [1]:
import os
import gc
import torch
from datasets import load_dataset
from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
import evaluate
import warnings

# Suppress warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=SyntaxWarning)

# Environment settings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
gc.collect()
torch.cuda.empty_cache()

/home/panha/miniconda3/envs/tranformer-xl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load dataset
dataset = load_dataset("Zedthecodex/KhmerTimes-Summary")

# Split train/validation
splits = dataset["train"].train_test_split(test_size=0.1, seed=42)
train_dataset = splits["train"]
val_dataset = splits["test"]

print(f"Train size: {len(train_dataset)}")
print(f"Validation size: {len(val_dataset)}")

Train size: 11588
Validation size: 1288


In [3]:
# Load mBART tokenizer and model
model_name = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
tokenizer.src_lang = "km_KH"
tokenizer.tgt_lang = "km_KH"

# Use bf16 for memory efficiency, gradient checkpointing
model = MBartForConditionalGeneration.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
model.gradient_checkpointing_enable()

`torch_dtype` is deprecated! Use `dtype` instead!


In [4]:
# Preprocessing
max_input_length = 384
max_target_length = 96

def preprocess(batch):
    inputs = [str(x) if x is not None else "" for x in batch["Article"]]
    targets = [str(x) if x is not None else "" for x in batch["Summary"]]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply preprocessing
tokenized_train = train_dataset.map(preprocess, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(preprocess, batched=True, remove_columns=val_dataset.column_names)

In [5]:
# Evaluation metric
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels)
    return {k: round(v * 100, 2) for k, v in result.items()}

In [7]:
# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./mbart-khmer",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    weight_decay=0.01,
    num_train_epochs=2,
    predict_with_generate=True,
    bf16=True,
    logging_strategy="steps",
    logging_steps=50,
    log_level="info",
    load_best_model_at_end=True,
    metric_for_best_model="eval_rougeL",
    report_to="none",
    remove_unused_columns=False
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [8]:
# Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

The model is already on multiple devices. Skipping the move to device specified in `args`.
Using auto half precision backend


In [9]:
# Start training
trainer.train()

***** Running training *****
  Num examples = 11,588
  Num Epochs = 2
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 8
  Total optimization steps = 2,898
  Number of trainable parameters = 610,879,488


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,0.463000,0.456928,5.340000,2.750000,5.280000,5.300000
2,0.417500,0.452793,5.670000,3.200000,5.640000,5.650000



***** Running Evaluation *****
  Num examples = 1288
  Batch size = 1
Generate config GenerationConfig {
  "bos_token_id": 0,
  "decoder_start_token_id": 2,
  "early_stopping": true,
  "eos_token_id": 2,
  "forced_eos_token_id": 2,
  "max_length": 200,
  "num_beams": 5,
  "pad_token_id": 1
}

Saving model checkpoint to ./mbart-khmer/checkpoint-1449
Configuration saved in ./mbart-khmer/checkpoint-1449/config.json
Configuration saved in ./mbart-khmer/checkpoint-1449/generation_config.json
Model weights saved in ./mbart-khmer/checkpoint-1449/model.safetensors
Saving Trainer.data_collator.tokenizer by default as Trainer.processing_class is `None`
tokenizer config file saved in ./mbart-khmer/checkpoint-1449/tokenizer_config.json
Special tokens file saved in ./mbart-khmer/checkpoint-1449/special_tokens_map.json

***** Running Evaluation *****
  Num examples = 1288
  Batch size = 1
Saving model checkpoint to ./mbart-khmer/checkpoint-2898
Configuration saved in ./mbart-khmer/checkpoint-2898/c

TrainOutput(global_step=2898, training_loss=0.5535745956389306, metrics={'train_runtime': 1566.5288, 'train_samples_per_second': 14.794, 'train_steps_per_second': 1.85, 'total_flos': 1.8834524813131776e+16, 'train_loss': 0.5535745956389306, 'epoch': 2.0})

In [10]:
# Save model and tokenizer
model.save_pretrained("./mbart-khmer")
tokenizer.save_pretrained("./mbart-khmer")

Configuration saved in ./mbart-khmer/config.json
Configuration saved in ./mbart-khmer/generation_config.json
Model weights saved in ./mbart-khmer/model.safetensors
tokenizer config file saved in ./mbart-khmer/tokenizer_config.json
Special tokens file saved in ./mbart-khmer/special_tokens_map.json


('./mbart-khmer/tokenizer_config.json',
 './mbart-khmer/special_tokens_map.json',
 './mbart-khmer/sentencepiece.bpe.model',
 './mbart-khmer/added_tokens.json',
 './mbart-khmer/tokenizer.json')

In [15]:
# Inference example
sample_text = val_dataset[0]["Article"]
inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, padding=True).to(model.device)
summary_ids = model.generate(**inputs, max_length=128)
print("\nOriginal article:\n", sample_text[:400], "...")
print("\nGenerated summary:\n", tokenizer.decode(summary_ids[0], skip_special_tokens=True))


Original article:
 កាលពី​ពេលថ្មីៗនេះ​ ​អាជ្ញាធរ​ខេត្ត​ព្រះ​សី​ហ​នុ ​បាន​ចុះ​បង្ក្រាប​ជនបរទេស​២​នាក់​ ​ដែល​ផលិត​វីដេអូ​ក្លែងក្លាយ​ ​ក្នុង​គោលបំណង​បំភាន់​សាធារណជន​អំពី​បញ្ហា​សន្តិសុខ​ក្នុង​ព្រះរាជាណាចក្រកម្ពុជា​ ​នេះ​បើ​យោង​តាម​លោក​ ​ហ៊ុន​ ​ម៉ា​នី​ ​រដ្ឋមន្ត្រី​ក្រសួង​មុខងារ​សាធារណៈ​ ​ដែល​បាន​បង្ហោះ​នៅ​លើ​ ​Face​book​ ​របស់​លោក​។​
​លោក​បាន​សរសេរ​នៅ​ថ្ងៃ​ទី​១​៧​ ​ខែកុម្ភៈ​ ​ឆ្នាំ​២​០​២​៤​ ​ថា​ ​នេះ​ជា​ឈុត​ឆាក​ដែល​ថត​ដោ ...

Generated summary:
 អាជ្ញាធរ ខេត្ត ព្រះ សី ហ នុ ចុះ បង្ក្រាប ជនបរទេស ២ នាក់ ដែល ផលិត វីដេអូ ក្លែងក្លាយ
